In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
#Define order configuration
SEED = 20261022
ORDER_COUNT = 4000

# Snapshot date used by RFM, segments and recommendations (notebooks 07-11)
ANALYSIS_DATE = "2026-10-30"
# Data runs 30 days past the snapshot so recommendation outcomes can be measured
DATA_END_DATE = "2026-11-29"

print("Seed:", SEED)
print("Total Orders:", ORDER_COUNT)

Seed: 20261022
Total Orders: 4000


########Generate order records

In [0]:

orders = spark.range(1, ORDER_COUNT + 1)

orders = orders.withColumn(
    "order_id",
    F.concat(
        F.lit("ORD-2026-"),
        F.lpad(F.col("id").cast("string"), 6, "0")
    )
)



In [0]:
# Select a customer using deterministic hashing

CUSTOMER_COUNT = 1000

orders = orders.withColumn(
    "customer_number",
    F.pmod(
        F.xxhash64("order_id", F.lit(SEED)),
        F.lit(CUSTOMER_COUNT)
    ) + 1
)

orders = orders.withColumn(
    "customer_id",
    F.concat(
        F.lit("CUS-"),
        F.lpad(F.col("customer_number").cast("string"), 6, "0")
    )
)



In [0]:
# Generate order dates: on or after the customer's signup date, up to DATA_END_DATE.
# Orders after ANALYSIS_DATE form the 30-day outcome window used in notebook 11.

customer_signup = spark.table("customer_master").select("customer_id", "signup_date")

orders = orders.join(customer_signup, on="customer_id", how="inner")

days_available = F.datediff(F.lit(DATA_END_DATE).cast("date"), F.col("signup_date")) + 1

orders = orders.withColumn(
    "order_date",
    F.date_add(
        F.col("signup_date"),
        F.pmod(F.xxhash64("order_id", F.lit("date")), days_available).cast("int")
    )
)


In [0]:
# Generate order status

orders = orders.withColumn(
    "order_status",
    F.when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 85, "DELIVERED")
     .when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 93, "RETURNED")
     .when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 97, "CANCELLED")
     .otherwise("PENDING")
)



In [0]:
# Generate payment methods

orders = orders.withColumn(
    "payment_method",
    F.when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 0, "UPI")
     .when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 1, "CARD")
     .when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 2, "NET_BANKING")
     .otherwise("COD")
)

order_df = orders.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_status",
    "payment_method"
)

display(order_df.limit(10))

order_id,customer_id,order_date,order_status,payment_method
ORD-2026-000001,CUS-000933,2026-09-10,DELIVERED,CARD
ORD-2026-000002,CUS-000771,2025-06-12,RETURNED,NET_BANKING
ORD-2026-000003,CUS-000624,2025-07-08,DELIVERED,UPI
ORD-2026-000004,CUS-000758,2026-07-28,DELIVERED,CARD
ORD-2026-000005,CUS-000433,2026-02-02,DELIVERED,COD
ORD-2026-000006,CUS-000346,2026-02-12,DELIVERED,NET_BANKING
ORD-2026-000007,CUS-000516,2025-11-25,DELIVERED,NET_BANKING
ORD-2026-000008,CUS-000606,2025-06-12,DELIVERED,CARD
ORD-2026-000009,CUS-000395,2026-06-11,DELIVERED,UPI
ORD-2026-000010,CUS-000396,2025-08-11,DELIVERED,COD


In [0]:
#Validate the order data
print("Total Orders:", order_df.count())

Total Orders: 4000


In [0]:
#Check duplicate order IDs:
order_df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
#Check customer foreign keys:
invalid_customers = order_df.join(
    spark.read.table("customer_master").select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Invalid Customer References:", invalid_customers.count())

Invalid Customer References: 0


In [0]:
#Check order statuses:
order_df.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   DELIVERED| 3432|
|     PENDING|  113|
|    RETURNED|  281|
|   CANCELLED|  174|
+------------+-----+



In [0]:
#Save the Order table
order_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("order_master")

In [0]:
%sql
SELECT * FROM order_master LIMIT 10;

order_id,customer_id,order_date,order_status,payment_method
ORD-2026-001001,CUS-000037,2026-09-18,DELIVERED,UPI
ORD-2026-001002,CUS-000802,2026-10-21,PENDING,CARD
ORD-2026-001003,CUS-000551,2026-02-05,DELIVERED,COD
ORD-2026-001004,CUS-000761,2026-02-04,DELIVERED,NET_BANKING
ORD-2026-001005,CUS-000315,2026-04-26,CANCELLED,NET_BANKING
ORD-2026-001006,CUS-000333,2026-11-16,DELIVERED,UPI
ORD-2026-001007,CUS-000958,2024-10-23,DELIVERED,NET_BANKING
ORD-2026-001008,CUS-000541,2025-10-19,RETURNED,NET_BANKING
ORD-2026-001009,CUS-000732,2026-03-22,DELIVERED,UPI
ORD-2026-001010,CUS-000317,2026-04-22,DELIVERED,CARD
